# Scalability experiments

The analysis below uses one validated long-form table for every plot. A CVaR timeout contributes a **capped wall time of 86,400 seconds** and is reported separately in the timeout rate. Success rates use all replicates; eta summaries use only successful, completed runs.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from matplotlib.lines import Line2D

EXPERIMENTS_DIR = Path(".") if Path("results").exists() else Path("experiments")
RESULTS_DIR = EXPERIMENTS_DIR / "results"
analysis_path = str(EXPERIMENTS_DIR.resolve())
if analysis_path not in sys.path:
    sys.path.insert(0, analysis_path)

import scalability_analysis as sa  # noqa: E402

K_GRID = [5, 10, 30, 100, 250, 500]
PLOT_K_GRID = [30, 100, 250, 500]
N_GRID = [5, 10, 20, 50]
RISK_GRID = ["msd", "cvar"]
METHODS = [*sa.SCALABILITY_METHODS, "higher_eps_screened_dual", "uniform"]
cmap = plt.get_cmap("tab10")
COLORS = {method: cmap(i % cmap.N) for i, method in enumerate(METHODS)}

In [ ]:
msd_wide = sa.load_csv_shards(
    RESULTS_DIR / "remote/msd_cvar_part_scalability", "*K*_n*.csv"
)
msd_wide = msd_wide.loc[msd_wide["risk"].ne("cvar")].copy()

cvar_capped = pd.read_csv(
    RESULTS_DIR / "remote/cvar_scalability/capped_24h_v1/capped_method_results.csv"
)
cvar_wide = sa.capped_results_to_wide(cvar_capped)

expected_rows_per_risk = len(K_GRID) * len(N_GRID) * 20
assert len(msd_wide) == expected_rows_per_risk
assert len(cvar_wide) == expected_rows_per_risk

# Keep the familiar one-row-per-replicate dataframe for interactive inspection.
df = pd.concat([msd_wide, cvar_wide], ignore_index=True, sort=False)
assert not df.duplicated(["risk", "K", "n", "seed"]).any()

higher_epsilon = sa.load_csv_shards(RESULTS_DIR / "remote/higher_eps/v1", "*K*_n*.csv")
df = sa.merge_higher_epsilon_screened_dual(df, higher_epsilon)

scalability_long = sa.wide_scalability_to_long(
    df, methods=[*sa.SCALABILITY_METHODS, "higher_eps_screened_dual"]
)
uniform = pd.read_csv(RESULTS_DIR / "uniform/uniform_profile_baseline.csv")
scalability_long = sa.append_uniform_baseline(scalability_long, uniform)
scalability_summary = sa.summarize_scalability(scalability_long)

coverage = (
    scalability_long.groupby(["risk", "method"], as_index=False)
    .agg(replicate_method_rows=("seed", "size"))
    .sort_values(["risk", "method"])
)
coverage

In [ ]:
scalability_summary.sort_values(["risk", "n", "K", "method"])

In [ ]:
def plot_scalability_grid(
    summary,
    metric,
    ylabel,
    *,
    q25=None,
    q75=None,
    yscale=None,
    ylim=None,
    mark_zero_success=False,
):
    fig, axes = plt.subplots(
        len(RISK_GRID),
        len(N_GRID),
        figsize=(15, 7),
        dpi=600,
        sharex=True,
        sharey=True,
    )

    for row, risk in enumerate(RISK_GRID):
        for col, n in enumerate(N_GRID):
            ax = axes[row, col]
            panel = summary.loc[
                summary["risk"].eq(risk)
                & summary["n"].eq(n)
                & summary["K"].isin(PLOT_K_GRID)
            ]

            for method in METHODS:
                points = panel.loc[panel["method"].eq(method)].sort_values("K")
                points = points.loc[points[metric].notna()]
                if points.empty:
                    continue

                ax.plot(
                    points["K"].to_numpy(),
                    points[metric].to_numpy(),
                    "-",
                    color=COLORS[method],
                )
                if q25 is not None and q75 is not None:
                    band = points.dropna(subset=[q25, q75])
                    ax.fill_between(
                        band["K"].to_numpy(),
                        band[q25].to_numpy(),
                        band[q75].to_numpy(),
                        alpha=0.15,
                        color=COLORS[method],
                    )

                zero_success = points["successes"].eq(0) if mark_zero_success else False
                if mark_zero_success:
                    ordinary = points.loc[~zero_success]
                    failed = points.loc[zero_success]
                else:
                    ordinary = points
                    failed = points.iloc[0:0]
                ax.scatter(
                    ordinary["K"],
                    ordinary[metric],
                    marker="o",
                    s=28,
                    zorder=4,
                    color=COLORS[method],
                )
                ax.scatter(
                    failed["K"],
                    failed[metric],
                    marker="x",
                    s=70,
                    linewidths=2.0,
                    zorder=5,
                    color=COLORS[method],
                )

            if yscale is not None:
                ax.set_yscale(yscale)
            if ylim is not None:
                ax.set_ylim(*ylim)
            ax.set_title(f"{risk.upper()}, n={n}")
            ax.set_xlabel("K")
            ax.set_xticks(PLOT_K_GRID)
            ax.grid(alpha=0.25)
        axes[row, 0].set_ylabel(ylabel)

    present = set(summary["method"])
    legend_methods = [method for method in METHODS if method in present]
    handles = [
        Line2D([0], [0], color=COLORS[method], marker="o", label=method)
        for method in legend_methods
    ]
    if mark_zero_success:
        handles.append(
            Line2D(
                [0],
                [0],
                color="black",
                marker="x",
                linestyle="none",
                label="zero successful reps",
            )
        )
    axes[0, -1].legend(
        handles=handles, frameon=False, bbox_to_anchor=(1.02, 1), loc="upper left"
    )
    plt.tight_layout()
    return fig, axes

A timeout is not an observed 24-hour completion. The runtime figure therefore says **capped wall time**; the next two figures separately show success and timeout rates.

In [ ]:
plot_scalability_grid(
    scalability_summary,
    "capped_time_median",
    "Median capped wall time (s)",
    q25="capped_time_q25",
    q75="capped_time_q75",
    yscale="log",
    mark_zero_success=True,
)
plt.show()

In [ ]:
plot_scalability_grid(
    scalability_summary,
    "success_rate",
    "Success rate (all reps)",
    ylim=(-0.02, 1.02),
)
plt.show()

In [ ]:
plot_scalability_grid(
    scalability_summary,
    "timeout_rate",
    "Timeout rate (all reps)",
    ylim=(-0.02, 1.02),
)
plt.show()

In [ ]:
plot_scalability_grid(
    scalability_summary,
    "eta_median",
    "Median eta (successful completed reps)",
    q25="eta_q25",
    q75="eta_q75",
    yscale="log",
)
plt.show()

# Hyperparameter tuning using pilots for stochastic algorithms

Each loaded shard and method defines one continuation trajectory. Running-best eta and improvement therefore carry across changes in $\kappa$ and $\tau$ between stages, while distinct shards and methods remain separate. The eta figures show one shared trajectory panel per risk, with color identifying the initial step-size experiment.

In [ ]:
V2_ROOT = RESULTS_DIR / "stochastic/continuation_shards/v2"
v2_summary, v2_history = sa.load_stochastic_shards(V2_ROOT)
v2_best_history = sa.prepare_best_eta_history(v2_history)
v2_improvement_history = sa.prepare_eta_improvement(v2_history)

In [ ]:
v2_summary.groupby(["K", "n", "entropy_kappa", "smoothing_tau", "step_size"])[
    [
        "full_batch_eta",
        "full_batch_best_objective",
        "full_batch_best_certificate_eta",
        "full_batch_best_certificate_iteration",
        "minibatch_eta",
        "minibatch_best_objective",
        "minibatch_best_certificate_eta",
        "minibatch_best_certificate_iteration",
    ]
].median()

In [ ]:
v2_summary[
    [
        column
        for column in v2_summary.columns
        if "eta" in column or "time" in column or "iteration" in column
    ]
    + ["K", "n", "entropy_kappa", "smoothing_tau", "step_size"]
]

In [ ]:
def plot_eta_history(prepared_history, K, n, method):
    h = prepared_history.loc[
        prepared_history["K"].eq(K)
        & prepared_history["n"].eq(n)
        & prepared_history["method"].eq(method)
    ]
    if h.empty:
        return None

    g = sns.relplot(
        data=h,
        x="iteration",
        y="best_eta",
        col="risk",
        hue="step_size",
        kind="line",
        estimator="median",
        errorbar=("pi", 50),
        palette="viridis",
        height=4,
        aspect=1.25,
        facet_kws={"sharex": True, "sharey": True},
    )
    for ax in g.axes.flat:
        ax.axhline(1e-3, color="black", linestyle="--", linewidth=1)
        ax.grid(alpha=0.2)
    g.set_axis_labels("Iteration", "Best certified eta")
    g.set_titles("risk={col_name}")
    g.fig.suptitle(f"{method}: K={K}, n={n}", y=1.02)
    return g


def plot_eta_improvement(prepared_history, K, n, method):
    h = prepared_history.loc[
        prepared_history["K"].eq(K)
        & prepared_history["n"].eq(n)
        & prepared_history["method"].eq(method)
    ]
    if h.empty:
        return None

    g = sns.relplot(
        data=h,
        x="iteration",
        y="eta_improvement_pct",
        col="risk",
        hue="step_size",
        kind="line",
        estimator="median",
        errorbar=("pi", 50),
        palette="viridis",
        height=4,
        aspect=1.25,
    )
    for ax in g.axes.flat:
        ax.axhline(0, color="black", linewidth=1)
        ax.grid(alpha=0.2)
    g.set_axis_labels("Iteration", "Improvement over initial eta (%)")
    g.set_titles("risk={col_name}")
    g.fig.suptitle(f"{method}: K={K}, n={n}", y=1.02)
    return g


def plot_eta_objective(history, K, n):
    h = history.loc[history["K"].eq(K) & history["n"].eq(n)].copy()
    if h.empty:
        return None

    # Preserve the original diagnostic exclusion, but apply it to the filtered data.
    cvar_residual_mask = (
        h["risk"].eq("cvar") & h["entropy_kappa"].eq(0.1) & h["residual_norm"].lt(0.6)
    )
    plot_data = h.loc[~cvar_residual_mask]
    if plot_data.empty:
        return None

    g = sns.relplot(
        data=plot_data,
        x="residual_norm",
        y="eta",
        row="risk",
        hue="entropy_kappa",
        style="method",
        col="step_size",
        kind="scatter",
        height=3.2,
        facet_kws={"sharex": False, "sharey": True},
    )
    for ax in g.axes.flat:
        ax.grid(alpha=0.2)
    g.set_titles(r"risk={row_name}, step_size=${col_name}$")
    g.fig.suptitle(f"K={K}, n={n}", y=1.02)
    return g

In [ ]:
v2_combinations = (
    v2_history[["K", "n", "method"]].drop_duplicates().sort_values(["method", "K", "n"])
)

for row in v2_combinations.itertuples(index=False):
    plot_eta_history(v2_best_history, K=row.K, n=row.n, method=row.method)
    plt.show()

In [ ]:
for row in v2_combinations.itertuples(index=False):
    plot_eta_improvement(v2_improvement_history, K=row.K, n=row.n, method=row.method)
    plt.show()

In [ ]:
for row in (
    v2_history[["K", "n"]]
    .drop_duplicates()
    .sort_values(["K", "n"])
    .itertuples(index=False)
):
    plot_eta_objective(v2_history, K=row.K, n=row.n)
    plt.show()

## v3 continuation pilots

In [ ]:
V3_ROOT = RESULTS_DIR / "stochastic/continuation_shards/v3"
v3_summary, v3_history = sa.load_stochastic_shards(V3_ROOT)

warning_shards = {
    path.stem
    for path in (V3_ROOT / "logs").glob("*.log")
    if "RuntimeWarning" in path.read_text()
}
v3_summary["had_runtime_warning"] = v3_summary["shard"].isin(warning_shards)

v3_long = sa.stochastic_summary_to_long(v3_summary)
v3_long["initial_step"] = v3_long["step_size"]
v3_long.groupby(["risk", "method", "initial_step"]).agg(
    reps=("eta", "size"),
    successes=("success", "sum"),
    min_eta=("eta", "min"),
    median_eta=("eta", "median"),
    max_eta=("eta", "max"),
    median_time_s=("time_s", "median"),
    median_iterations=("iterations", "median"),
    warning_rate=("had_runtime_warning", "mean"),
)

In [ ]:
g = sns.relplot(
    data=v3_long,
    x="initial_step",
    y="eta",
    row="method",
    col="risk",
    hue="seed",
    units="seed",
    estimator=None,
    marker="o",
    kind="line",
    height=3,
)
for ax in g.axes.flat:
    ax.axhline(1e-3, color="black", linestyle="--")
    ax.grid(alpha=0.2)
g.set_axis_labels("Initial step size", "Best certified eta")

In [ ]:
v3_history_prepared = sa.prepare_v3_plot_history(v3_history, v3_summary)

g = sns.relplot(
    data=v3_history_prepared,
    x="iteration",
    y="best_eta_so_far",
    row="method",
    col="risk",
    hue="initial_step",
    kind="line",
    estimator="median",
    errorbar=("pi", 50),
    height=5,
)
for ax in g.axes.flat:
    ax.grid(alpha=0.2)